In [2]:
## CLEAN DATA functions
import re
import pandas as pd

# add new entry into jobs table
def update_job_table(job_props, parsed_salary, job_table, job_id):
    new_entry = {
        'JOB_ID': job_id,
        'TITLE': job_props[0],
        'COMPANY': job_props[1],
        'LOCATION': job_props[2],
        'EMPLOYMENT_TYPE': job_props[3],
        'SALARY': parsed_salary[0],
        'PAY_PERIOD': parsed_salary[1],
        'POST_DATE': job_props[5]
    }
    
    # Ensure that new_entry is a DataFrame (not a list)
    new_entry_df = pd.DataFrame([new_entry])  # Create DataFrame from the dictionary
    for row in new_entry_df.iloc[:, 1:].values:
        print(" | ".join(map(str, row))) # print results

    # Add the new entry to the job table
    if job_table.empty:
        return new_entry_df
    else:
        job_table = pd.concat([job_table, new_entry_df], ignore_index=True)
        return job_table



# add new entry into jobs table
def update_job_skills_table(job_props, parsed_salary, job_skills_table, df_skill_ids, job_id):
    
    # Add new entry for each skill
    for skill_id in df_skill_ids.skill_id:
        job_skills_data = {
            'job_id': job_id,
            'skill_id': skill_id
        }

        # add entry to datafram
        new_entry = pd.DataFrame([job_skills_data])

        if job_skills_table.empty:
            return new_entry
        else:
            job_skills_table = pd.concat([job_skills_table,  new_entry], ignore_index=True)
    return job_skills_table
    

# Clean up salary format
def parse_salary(salary):

    # Salaries without numbers -> None
    if salary is None or not re.search(r"\d", str(salary)):  # Check if there are no numbers
        return None, None

    # Remove commas for easier conversion
    salary = salary.replace(",", "")  

    # Match salary range (e.g., "$140,000 – $160,000 per year")
    match = re.search(r"(\d+[kK]?)\s*(?:–|-|to)\s*(\d+[kK]?)", salary)
    if match:
        low, high = match.groups()
    else:
        # Match single salary amount (e.g., "$170,000 + super")
        match = re.search(r"(\d+[kK]?)", salary)
        if match:
            low = match.group(1)
            high = low  # If only one value, use it for both low and high
        else:
            return None, None

    # Convert values to integers
    def to_int(value):
        return int(value.lower().replace("k", "000")) if "k" in value.lower() else int(value)

    low, high = to_int(low), to_int(high)
    avg_salary = (low + high) // 2  # Take the average

    # Determine pay period
    if "per year" in salary or "annually" in salary or "K + Super" in salary or "K + super" in salary:
        pay_period = "annually"
    elif "per day" in salary or "p.d." in salary:
        pay_period = "daily"
    elif "per week" in salary:
        pay_period = "weekly"
    elif "per month" in salary:
        pay_period = "monthly"
    else:
        pay_period = "annually"  # Default to annually if unclear

    return avg_salary, pay_period